#Word Generation Pipeline
This colab notebook downloads files from histwords and filters it. Then it determines low and high shift words. Taking a random sample from each. Then it uses those words to generate semantic neighbors.

##Setup

In [1]:
!mkdir embeddings
!cd embeddings
#!curl -o eng-fiction-all_sgns.zip http://snap.stanford.edu/historical_embeddings/eng-fiction-all_sgns.zip
#!unzip eng-fiction-all_sgns.zip
#!mv sgns eng-fiction-all_sgns

!curl -o eng-fiction-all.zip http://snap.stanford.edu/historical_embeddings/eng-fiction-all.zip
!unzip eng-fiction-all.zip


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2503M  100 2503M    0     0  38.0M      0  0:01:05  0:01:05 --:--:-- 40.8M
Archive:  eng-fiction-all.zip
   creating: eng-fiction-all/
   creating: eng-fiction-all/netstats/
  inflating: eng-fiction-all/netstats/full-nstop_nproper-top10000.pkl  
   creating: eng-fiction-all/svd/
  inflating: eng-fiction-all/svd/1860-vocab.pkl  
  inflating: eng-fiction-all/svd/1900-vocab.pkl  
  inflating: eng-fiction-all/svd/1860-w.npy  
  inflating: eng-fiction-all/svd/1900-w.npy  
  inflating: eng-fiction-all/svd/1840-w.npy  
  inflating: eng-fiction-all/svd/1920-w.npy  
  inflating: eng-fiction-all/svd/1890-w.npy  
  inflating: eng-fiction-all/svd/1970-vocab.pkl  
  inflating: eng-fiction-all/svd/1810-vocab.pkl  
  inflating: eng-fiction-all/svd/1940-w.npy  
  inflating: eng-fiction-all/svd/1920-vocab.pkl  
  inflating: eng-fiction-all/svd

In [2]:
!pip install --upgrade gensim
!pip install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 60.9 MB/s eta 0:00:00


In [3]:
import numpy as np
import pickle as p
from openai import OpenAI
from google.colab import userdata
from gensim.models import KeyedVectors
import pandas as pd
import random
from google.colab import drive
drive.mount('content/drive')

Find minimum frequency words.

In [4]:
with open(f"/content/eng-fiction-all/word_lists/full-nstop_nproper.pkl", "rb") as file:
    clean_words_sorted = p.load(file, encoding="latin1")

#with open(f"/content/eng-fiction-all/freqs.pkl", "rb") as file:
#    freq_of_words = p.load(file, encoding="latin1")

with open(f"/content/eng-fiction-all/volstats/vols.pkl", "rb") as file:
    score_of_words = p.load(file, encoding="latin1")

In [5]:
selected_decades = [decade for decade in range(1950, 1990+10, 10)]
print(selected_decades)

[1950, 1960, 1970, 1980, 1990]


In [6]:
top_freq_words = clean_words_sorted[:10000]
sanitized_top_freq_words = top_freq_words

In [7]:
for word in score_of_words:
  for decade in selected_decades:
    if np.isnan(score_of_words[word][decade]):
      if word in sanitized_top_freq_words:
        sanitized_top_freq_words.remove(word)

In [8]:
def avg_shift_distance(key):
  cumulative_distance_shift = 0
  for decade in selected_decades:
    cumulative_distance_shift += np.arccos(np.clip(score_of_words[key][decade],-1, 1))
  avg_shift_distance = cumulative_distance_shift/len(selected_decades)
  return avg_shift_distance

In [9]:
shift_list = []

for word in sanitized_top_freq_words:
  shift_list.append([word, avg_shift_distance(word)])

shift_list = sorted(shift_list,key=lambda x: x[1], reverse=True)


In [11]:
percentile = 0.25

low_shift_words = shift_list[int(len(shift_list)-len(shift_list)*percentile):]
high_shift_words = shift_list[:int(len(shift_list)*percentile)]

In [88]:
seed = 1042
num_words = 50

random.seed(seed)
low_target_word = random.sample(low_shift_words, num_words)
high_target_word = random.sample(high_shift_words, num_words)

In [14]:
models = []
count = 0
for decade in selected_decades:
  model = [decade]
  vectors = np.load(f"/content/eng-fiction-all/sgns/{decade}-w.npy")
  with open(f"/content/eng-fiction-all/sgns/{decade}-vocab.pkl", "rb") as file:
    keys = p.load(file)

  model.append(keys)
  model.append(KeyedVectors(vector_size=vectors.shape[1]))

  model[2].add_vectors(keys, vectors)
  models.append(model)

##Model Testing

In [22]:
client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

In [70]:
general_instructions = """
You are generating semantic neighborhoods in English.

Return exactly 10 single-word semantic neighbors for the target word.
Return a the words via comma seperation with no spacing.

Additional Rules:
- Lowercase only
- Single Words only
- No target word in output
- For the output no explanations, numbering, or punctuation inside the words.
- For the output exclude stop-words and proper nouns
"""

def prompt(word, pType, decade=None):
  if pType == "ahistorical":
    return f"""
    Target Word: {word}

    Task: Generate 10 semantic neighbors for the target word.
    """
  elif pType == "modern":
    return f"""
    Target Word: {word}

    Task: Generate 10 semantic neighbors for the target word as it is commonly used in modern English.
    """
  elif pType == "historical":
    return f"""
    Target Word: {word}

    Task: Generate 10 semantic neighbors for the target word as it is commonly used in historical English.
    """
  elif pType == "historical_decade":
    return f"""
    Target Word: {word}

    Task: Generate 10 semantic neighbors for the target word as it is commonly used in historical {decade}'s English.
    """


In [71]:
def chatGPT_response(word, decade, prompt):
    response = client.responses.create(
      model="gpt-5-nano-2025-08-07",
      instructions=general_instructions,
      input=prompt
    )
    return response.output_text.split(",")

def sanitize_list(output_list, model):
  t10ChatGPT = []
  for word in output_list:
    if word in model[1] and np.mean(model[2][word]) != 0:
      t10ChatGPT.append(word)
  return t10ChatGPT

def cosine_similarity(target_word, generated_words, model):
  target_vector = model[2][target_word]

  generated_vectors = [model[2][word] for word in generated_words]

  cos_sim_list = model[2].cosine_similarities(target_vector, generated_vectors)

  return cos_sim_list, np.mean(cos_sim_list)



#def chatGPT_Multiple(word, decade)

In [72]:
word_generations_high = {}
for word, score in high_target_word:
  multi_info = {}
  for count in range(3):
    info = {}
    info["ahistorical"] = chatGPT_response(word, decade, prompt(word, "ahistorical"))
    info["modern"] = chatGPT_response(word, decade, prompt(word, "modern"))
    info["historical"] = chatGPT_response(word, decade, prompt(word, "historical"))
    for decade in selected_decades:
      info[f"{decade}"] = chatGPT_response(word, decade, prompt(word, "historical_decade", decade))
    multi_info[count] = info
  word_generations_high[word] = multi_info



In [73]:
word_generations_low = {}
for word, score in low_target_word:
  multi_info = {}
  for count in range(3):
    info = {}
    info["ahistorical"] = chatGPT_response(word, decade, prompt(word, "ahistorical"))
    info["modern"] = chatGPT_response(word, decade, prompt(word, "modern"))
    info["historical"] = chatGPT_response(word, decade, prompt(word, "historical"))
    for decade in selected_decades:
      info[f"{decade}"] = chatGPT_response(word, decade, prompt(word, "historical_decade", decade))
    multi_info[count] = info
  word_generations_low[word] = multi_info



In [85]:
with open("word_generations_high.pkl", "wb") as file:
  p.dump(word_generations_high, file)
with open("word_generations_low.pkl", "wb") as file:
  p.dump(word_generations_low, file)